## langchain 기초


In [ ]:
import os
import pathlib
import sys

# 노트북이 어느 위치에서 실행되든 backend/app 이 들어 있는 폴더(hanwha-agent)를 찾아 루트로 삼는다
# - 폴더 이름(day02)에 기대지 않으므로 다른 날짜 노트북에 복사해도 그대로 쓸 수 있다
here = pathlib.Path.cwd().resolve()
candidates = [here, *here.parents, here / "hanwha-agent"]
ROOT = next((p for p in candidates if (p / "backend" / "app").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"hanwha-agent 루트를 찾지 못했습니다. 현재 위치: {here}")

os.chdir(ROOT)                                  # 상대경로(.env 등)의 기준
SANDBOX = ROOT / "sandbox" / "w4" / "day02"

# app 패키지를 import 할 수 있게 backend 를 모듈 검색 경로 맨 앞에 넣는다
# - os.chdir 만으로는 import 경로가 바뀌지 않는다
BACKEND = str(ROOT / "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

print("프로젝트 루트  :", ROOT)

In [4]:
from langchain_anthropic import ChatAnthropic
from app.core.config import get_settings

settings = get_settings()

# Claude ChatModel 생성
# pydantic-settings 는 .env 를 읽어 Settings 객체에만 담고
# os.environ 에는 넣지 않으므로 api_key 를 직접 넘겨준다
llm = ChatAnthropic(
    model = "claude-haiku-4-5",
    max_tokens = 500,
    api_key = settings.anthropic_api_key.get_secret_value(),
)

response = llm.invoke(
    "RAG가 무엇인지 한 문장으로 설명해주세요."
)

print(response)

content='# RAG (Retrieval-Augmented Generation)\n\n**외부 데이터베이스에서 관련 정보를 검색해 불러온 후, 이를 바탕으로 AI 모델이 더 정확하고 최신의 답변을 생성하는 기술입니다.**' additional_kwargs={} response_metadata={'id': 'msg_011CfHw3qGRDyrDEnGjx6RcF', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 31, 'output_tokens': 89, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--01a0c79b-0bfd-7342-88f9-ce1caaaf6673-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 31, 'output_tokens': 89, 'total_tokens': 120, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h

In [ ]:
llm.invoke(
    "RAG 이/가 무엇인지 비전공자에게 설명해주세요."
)
# RAG 무엇인지 설명해주세요.
# Embedding 무엇인지 설명해주세요.
# Vector DB 무엇인지 설명해주세요.
# -> 공통된 부분은 {변수}로 치환해서 템플릿처럼 prompt를 관리할 수 있다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Prompt 만들기
# - {topic} 자리만 바꿔가며 같은 문장을 재사용한다
prompt = ChatPromptTemplate.from_template(
	"{topic}에 대해 비전공자도 이해할 수 있도록 쉽게 설명해주세요."
)

# 2. LLM 은 위 셀에서 만든 것을 그대로 쓴다
'''
llm = ChatAnthropic(
    model = "claude-haiku-4-5",
    max_tokens = 500,
    api_key = settings.anthropic_api_key.get_secret_value(),
)
'''

# 3. AIMessage 에서 문자열만 꺼내는 파서
parser = StrOutputParser()

# 4. 파이프로 연결한다 (LCEL)
chain = prompt | llm | parser

# 5. 실행
# - invoke 에 넘기는 것은 딕셔너리 하나다
#   chain.invoke("topic": "RAG") 처럼 중괄호를 빼면 SyntaxError 가 난다
response = chain.invoke({"topic": "RAG"})

# 6. 파서를 거쳤으므로 AIMessage 가 아니라 str 이 나온다
print(response)
print(type(response))

- 가짜 모델 : 돈 들이지 않고 체인을 확인한다

In [2]:
# 여기서부터는 가짜 모델을 쓴다 (호출이 나가지 않아 돈이 들지 않고, 결과가 항상 같다)
# - langchain-core 가 공식으로 주는 테스트용 모델들이다
from langchain_core.language_models.fake_chat_models import (
    FakeListChatModel,       # 답 목록을 돌려가며 내놓는다 — invoke · batch 확인용
    GenericFakeChatModel,    # AIMessage 이터레이터 — 스트리밍 확인용
    ParrotFakeChatModel,     # 받은 입력을 그대로 돌려준다 — 프롬프트가 어떻게 조립됐는지 확인용
)

fake = FakeListChatModel(responses=["첫 번째 답", "두 번째 답"])

print("가짜 모델도 AIMessage 를 돌려준다 :", type(fake.invoke("아무 질문")).__name__)

가짜 모델도 AIMessage 를 돌려준다 : AIMessage


- Output Parser 가 하는 일

In [3]:
# 파서가 있고 없고의 차이
chain_raw = prompt | fake                    # AIMessage 가 그대로 나온다
chain_str = prompt | fake | StrOutputParser()  # content 만 꺼내 str 로 나온다

raw = chain_raw.invoke({"topic": "RAG"})
txt = chain_str.invoke({"topic": "RAG"})

print("파서 없음 :", type(raw).__name__, "→", repr(raw.content))
print("파서 있음 :", type(txt).__name__, "→", repr(txt))

# type() 에 TextAccessor 라고 찍히지만 str 의 하위 클래스다 — 문자열처럼 그대로 쓰면 된다
print("str 인가   :", isinstance(txt, str))

파서 없음 : AIMessage → '두 번째 답'
파서 있음 : TextAccessor → '첫 번째 답'
str 인가   : True


- system / human 분리

In [4]:
# system 과 human 을 나눠서 만들기
# - system : 역할·규칙처럼 매 요청 같은 것
# - human  : 매 요청 달라지는 것
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "너는 IT 박사님이야. 초보자가 이해하기 쉽게 설명해줘"),
    ("human", "{question}"),
])

# 조립 결과를 그대로 보려면 프롬프트만 invoke 한다 (모델을 부르지 않는다)
msgs = prompt2.invoke({"question": "RAG가 뭔가요?"})
for m in msgs.messages:
    print(f"  {m.type:<7} {m.content}")

# ParrotFakeChatModel 은 "마지막 메시지"를 그대로 돌려준다
# - 전체 프롬프트가 아니라 human 메시지만 나온다. 체인이 무엇을 모델에 넘겼는지 확인할 때 쓴다
parrot = ParrotFakeChatModel()
print()
print("Parrot 응답 :", repr((prompt2 | parrot).invoke({"question": "RAG가 뭔가요?"}).content))

  system  너는 IT 박사님이야. 초보자가 이해하기 쉽게 설명해줘
  human   RAG가 뭔가요?

Parrot 응답 : 'RAG가 뭔가요?'


- batch

In [5]:
# batch : 여러 입력을 한 번에 넣는다 (for 문으로 하나씩 도는 것보다 낫다)
chain_fake = prompt | fake | StrOutputParser()

print("invoke :", chain_fake.invoke({"topic": "RAG"}))
print("batch  :", chain_fake.batch([
    {"topic": "RAG"},
    {"topic": "Embedding"},
    {"topic": "Vector DB"},
]))

# FakeListChatModel 은 목록을 "돌려가며" 내놓는다
# - 답이 2개뿐인데 3건을 물으면 첫 번째로 돌아간다. 실제 모델과 헷갈리지 않게 알아둔다

invoke : 두 번째 답
batch  : ['첫 번째 답', '두 번째 답', '첫 번째 답']


- stream

In [6]:
from langchain_core.messages import AIMessage

# stream : 다 만들어질 때까지 기다리지 않고 오는 대로 받는다
# - 답이 길 때 사용자가 기다리는 느낌을 줄인다
streamer = GenericFakeChatModel(messages=iter([
    AIMessage(content="RAG 는 검색해서 가져온 근거를 프롬프트에 붙여 답하게 하는 방식입니다."),
]))

for chunk in (prompt | streamer | StrOutputParser()).stream({"topic": "RAG"}):
    print(chunk, end="", flush=True)
print()
print("(조각으로 나뉘어 도착한다)")

RAG 는 검색해서 가져온 근거를 프롬프트에 붙여 답하게 하는 방식입니다.
(조각으로 나뉘어 도착한다)


- RunnableLambda · RunnablePassthrough

In [7]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 직접 만든 함수를 체인에 끼우려면 RunnableLambda 로 감싼다
def lookup(payload: dict) -> str:
    known = {"DOC-HR-014", "DOC-PU-007", "DOC-SE-003"}
    if payload["doc_id"] not in known:
        raise ValueError("모르는 문서...")
    return f"{payload['doc_id']} 조회 완료"

lookup_chain = RunnableLambda(lookup)
print("조회 :", lookup_chain.invoke({"doc_id": "DOC-HR-014"}))

# RunnablePassthrough : 받은 값을 그대로 흘려보낸다
# - RAG 에서 질문 하나를 두 갈래로 쓴다. 한쪽은 검색에, 다른 한쪽은 프롬프트에 그대로
#   딕셔너리로 묶으면 두 갈래가 동시에 실행되고 결과가 하나의 딕셔너리로 합쳐진다
retrieve = RunnableLambda(lambda q: f"[검색결과] {q} → 국내출장 여비 규정 제12조")

fan_out = {
    "context": retrieve,
    "question": RunnablePassthrough(),
}

from langchain_core.runnables import RunnableParallel
result = RunnableParallel(fan_out).invoke("부산 출장 숙박비")
print("context  :", result["context"])
print("question :", result["question"])

조회 : DOC-HR-014 조회 완료
context  : [검색결과] 부산 출장 숙박비 → 국내출장 여비 규정 제12조
question : 부산 출장 숙박비
